Fridge Ingredient Detection — YOLOv8n Training
Kaggle Notebook 1 | T14.1 SMAI Assignment 3
Dataset : shubhadeepmandal/fridge-object-dataset  (Fridge-Object v3, 53 classes)
Hardware : Kaggle GPU T4 x2  |  100 epochs  |  patience 5
─────────────────────────────────────────────────────────────

In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────
!pip install -q ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 115.7 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────
import os
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display, Image

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
WORK_DIR    = Path("/kaggle/working")
# ── Dataset uploaded to Kaggle as: shubhadeepmandal/fridge-object-dataset
DATASET_DIR = Path("/kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3")

── Cell 3: Verify dataset ────────────────────────────────────
The dataset is already available at DATASET_DIR (Kaggle input).
data.yaml was shipped with the Roboflow export — just load it.

In [4]:
yaml_path = DATASET_DIR / "data.yaml"
assert yaml_path.exists(), f"data.yaml not found at {yaml_path}"

In [5]:
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

In [6]:
# Patch absolute paths so YOLO can find images inside /kaggle/input
data_cfg["path"] = str(DATASET_DIR)
data_cfg["train"] = "train/images"
data_cfg["val"]   = "valid/images"
data_cfg["test"]  = "test/images"

In [7]:
# Write patched yaml to /kaggle/working (read-only input can't be edited)
patched_yaml = WORK_DIR / "data.yaml"
with open(patched_yaml, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

In [8]:
CLASS_NAMES = data_cfg["names"]
print(f"Dataset   : {DATASET_DIR}")
print(f"Classes   : {data_cfg['nc']} — {CLASS_NAMES[:6]} ...")
print(f"Train imgs: {len(list((DATASET_DIR/'train'/'images').glob('*')))}")
print(f"Val imgs  : {len(list((DATASET_DIR/'valid'/'images').glob('*')))}")
print(f"Test imgs : {len(list((DATASET_DIR/'test' /'images').glob('*')))}")

Dataset   : /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3
Classes   : 53 — ['apple', 'asparagus', 'avocado', 'banana', 'bean', 'beef'] ...
Train imgs: 5502
Val imgs  : 3040
Test imgs : 1331


In [9]:
# ── Cell 5: Training ──────────────────────────────────────────
model = YOLO("yolov8n.pt")  # nano — ~6 MB, fast inference on CPU/mobile

In [10]:
RUN_NAME = "fridge_yolov8n"

In [11]:
results = model.train(
    data    = str(patched_yaml),
    epochs  = 100,          # full training budget
    imgsz   = 640,
    batch   = 32,           # fits T4 16 GB; lower to 16 if OOM
    device  = 0,            # single GPU T4; use [0,1] for T4x2
    patience= 5,            # early-stop after 5 epochs no improvement
    project = str(WORK_DIR / "runs"),
    name    = RUN_NAME,
    exist_ok= True,
    # Kitchen-specific augmentations
    hsv_h       = 0.015,   # hue shift — different lighting
    hsv_s       = 0.7,     # saturation — fresh vs. wilted produce
    hsv_v       = 0.4,     # brightness — fridge dark vs. counter bright
    degrees     = 10,      # slight rotation
    fliplr      = 0.5,     # horizontal flip
    flipud      = 0.0,     # no vertical flip
    mosaic      = 1.0,     # group 4 ingredients on same canvas
    mixup       = 0.1,     # blend two images slightly
    copy_paste  = 0.1,     # copy-paste augmentation
    cache       = True,    # cache images in RAM for speed
    workers     = 4,
    verbose     = True,
)

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fridge_yolov8n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, p

In [12]:
# ── Cell 6: Evaluation ────────────────────────────────────────
best_model_path = WORK_DIR / "runs" / RUN_NAME / "weights" / "best.pt"
eval_model = YOLO(str(best_model_path))

In [13]:
metrics = eval_model.val(data=str(patched_yaml))

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,015,983 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 219.8±127.8 MB/s, size: 119.8 KB)
val: Scanning /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3/valid/labels... 3040 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3040/3040 1.1Kit/s 2.8s0.1ss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3/valid is not writable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 30642. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 190/190 7.3it/s 26.0s0.1ss


In [14]:
print(f"\n{'='*50}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")
print(f"{'='*50}\n")


mAP50:    0.6777
mAP50-95: 0.4951
Precision:0.7951
Recall:   0.6449



In [15]:
# Per-class AP50
print("Per-class AP50:")
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f"  {name:20s}: {metrics.box.ap50[i]:.3f}")

Per-class AP50:
  apple               : 0.497
  asparagus           : 0.000
  avocado             : 0.011
  banana              : 0.696
  bean                : 0.964
  beef                : 0.988
  blueberry           : 0.971
  bread               : 0.803
  broccoli            : 0.685
  butter              : 0.956
  cabbage             : 0.983
  carrot              : 0.878
  cauliflower         : 0.709
  cheese              : 0.747
  chicken breast      : 0.995
  chicken             : 0.969
  chilli              : 0.000
  chocolate           : 0.951
  corn                : 0.976
  cucumber            : 0.057
  egg                 : 0.548
  eggplant            : 0.815
  fish                : 0.000
  flour               : 0.995
  garlic              : 0.749
  ginger              : 0.833
  grape               : 0.757
  ham                 : 0.995
  kiwi                : 0.827
  leek                : 0.473
  lemon               : 0.280
  lettuce             : 0.798
  lime                : 

In [ ]:
# ── Cell 7: Speed Benchmark ───────────────────────────────────
benchmark_results = eval_model.benchmark(
    data   = str(patched_yaml),
    imgsz  = 640,
    half   = False,
    device = 0,
)
print(benchmark_results)

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 178.1±61.9 MB/s, size: 73.0 KB)
val: Scanning /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3/valid/labels... 3040 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3040/3040 1.0Kit/s 2.9s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3/valid is not writable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 28, len(boxes) = 30642. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3040/3040 75.2it/s 40.4s<0.0s
                   all       3040      30642      0.795      0.645      0.678      0.495
Speed: 1.1ms preprocess, 7.3ms inference, 0.0ms loss, 1.1ms postprocess pe

E0000 00:00:1776249240.510508      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776249240.583599      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776249241.153383      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776249241.153415      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776249241.153429      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776249241.153432      55 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 1.31s
Prepared 5 packages in 253ms
Installed 5 packages in 9ms
 + ai-edge-litert==2.1.4
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + onnx2tf==1.28.8
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 1.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 18...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 1.6s, saved as '/kaggle/working/runs/fridge_yolov8n/weights/best.onnx' (11.7 MB)
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /kaggle/working/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 46.6files/s 0.0s
Te

I0000 00:00:1776249267.256024      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12829 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776249270.977078      55 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at '/kaggle/working/runs/fridge_yolov8n/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 57, 8400), dtype=tf.float32, name=None)
Captures:
  133555708456080: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133555708454928: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  133555708455312: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  133555708459536: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133555708460688: TensorSpec(shape=(3, 3, 16, 32), dtype=tf.float32, name=None)
  133555708460304: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  133555708456272: TensorSpec(shape=(1, 1, 32, 32), dtype=tf.float32, name=None)
  133555708460880: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  133555706381904: TensorSpec(shape=(4,), dtype=tf.int64, name=No

...

I0000 00:00:1776249366.991207      55 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1776249366.991377      55 single_machine.cc:374] Starting new session
I0000 00:00:1776249366.998936      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 12829 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


TensorFlow GraphDef: export success ✅ 1.0s, saved as '/kaggle/working/runs/fridge_yolov8n/weights/best.pb' (11.7 MB)

Export complete (11.2s)
Results saved to /kaggle/working/runs/fridge_yolov8n/weights
Predict:         yolo predict task=detect model=/kaggle/working/runs/fridge_yolov8n/weights/best.pb imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/fridge_yolov8n/weights/best.pb imgsz=640 data=/kaggle/working/data.yaml  
Visualize:       https://netron.app
Loading /kaggle/working/runs/fridge_yolov8n/weights/best.pb for TensorFlow GraphDef inference...
Loading /kaggle/working/runs/fridge_yolov8n/weights/best.pb for TensorFlow GraphDef inference...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 205.5±149.4 MB/s, size: 98.7 KB)
val: Scanning /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3/valid/labels... 3040 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3040/3040 1.1Kit/s 2

pnnxparam = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cuda
inputshape = [1,3,640,640]f32
inputshape2 = 
customop = 
moduleop = 
############# pass_level0
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Bottleneck
inline module = ultralytics.nn.modules.block.C2f
inline module = ultralytics.nn.modules.block.DFL
inline module = ultralytics.nn.modules.block.SPPF
inline module =

NCNN: export success ✅ 4.4s, saved as '/kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model' (11.7 MB)

Export complete (4.4s)
Results saved to /kaggle/working/runs/fridge_yolov8n/weights
Predict:         yolo predict task=detect model=/kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model imgsz=640 data=/kaggle/working/data.yaml  
Visualize:       https://netron.app
Loading /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model for NCNN inference...
Loading /kaggle/working/runs/fridge_yolov8n/weights/best_ncnn_model for NCNN inference...
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 169.3±62.7 MB/s, size: 106.7 KB)
val: Scanning /kaggle/input/datasets/shubhadeepmandal/fridge-object-dataset/Fridge-Object-3/valid/labels... 3040 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3040/3040 1.0Kit/s 3.0s<0.

Statistics Collection: 1834it [30:26,  7.99s/it]

In [1]:
# ── Cell 8: Export to ONNX (for Streamlit Python inference) ──
onnx_path = eval_model.export(format="onnx", imgsz=640, opset=12)
print(f"ONNX model saved: {onnx_path}")

NameError: name 'eval_model' is not defined

── Cell 9: Export to TFLite INT8 (for React Native mobile) ──
tflite_path = eval_model.export(format="tflite", int8=True, imgsz=640)
print(f"TFLite INT8 model saved: {tflite_path}")
NOTE: TFLite export sometimes fails on Kaggle — uncomment only if needed.

In [ ]:
#── Cell 9: Export to TFLite INT8 (for React Native mobile) ──
tflite_path = eval_model.export(format="tflite", int8=True, imgsz=640)
print(f"TFLite INT8 model saved: {tflite_path}")
NOTE: TFLite export sometimes fails on Kaggle — uncomment only if needed.

In [ ]:
# ── Cell 10: Training Curves ──────────────────────────────────
results_csv = WORK_DIR / "runs" / RUN_NAME / "results.csv"
import pandas as pd
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("YOLOv8n Training Curves — Fridge Ingredient Detection (53 classes)", fontsize=14)

In [ ]:
metrics_to_plot = [
    ("train/box_loss",   "Train Box Loss",  "orange"),
    ("train/cls_loss",   "Train Cls Loss",  "red"),
    ("metrics/mAP50",    "mAP@0.50",        "green"),
    ("metrics/mAP50-95", "mAP@0.50:0.95",   "blue"),
    ("metrics/precision","Precision",       "purple"),
    ("metrics/recall",   "Recall",          "teal"),
]

In [ ]:
for ax, (col, label, color) in zip(axes.flat, metrics_to_plot):
    if col in df.columns:
        ax.plot(df["epoch"], df[col], color=color, linewidth=2)
        ax.set_title(label)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig(str(WORK_DIR / "training_curves.png"), dpi=150)
plt.show()

In [ ]:
print("\n✅ Training complete! See Cell 11 below to download your files.")

── Cell 11: Download outputs ─────────────────────────────────────────────────
Bundles best.pt, best.onnx, training_curves.png into a single zip and
shows clickable FileLinks directly in the Kaggle notebook output panel.

In [ ]:
import shutil
from IPython.display import FileLink, display

In [ ]:
WEIGHTS_DIR = WORK_DIR / "runs" / RUN_NAME / "weights"

In [ ]:
files_to_bundle = {
    "best.pt"            : WEIGHTS_DIR / "best.pt",
    "best.onnx"          : WEIGHTS_DIR / "best.onnx",
    "training_curves.png": WORK_DIR / "training_curves.png",
    "data.yaml"          : patched_yaml,
}

In [ ]:
bundle_dir = WORK_DIR / "download_bundle"
bundle_dir.mkdir(exist_ok=True)

In [ ]:
print("Bundling files:")
for fname, src in files_to_bundle.items():
    if src.exists():
        shutil.copy2(src, bundle_dir / fname)
        print(f"  [OK]     {fname}  ({src.stat().st_size / 1e6:.2f} MB)")
    else:
        print(f"  [SKIP]   {fname}  (not found)")

In [ ]:
# Create zip archive
zip_out = WORK_DIR / "fridge_yolov8n_outputs"
shutil.make_archive(str(zip_out), "zip", str(bundle_dir))
print(f"\nAll-in-one zip: {zip_out}.zip  ({(zip_out.with_suffix('.zip')).stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Clickable links
print("\n=== Click to download ===")
display(FileLink(str(zip_out) + ".zip",
                 result_html_prefix="<b>📦 All outputs (zip):</b> &nbsp;"))

In [ ]:
for fname, src in files_to_bundle.items():
    dest = bundle_dir / fname
    if dest.exists():
        display(FileLink(str(dest),
                         result_html_prefix=f"&nbsp;&nbsp;&nbsp;{fname}: "))

In [ ]:
print("\nAfter download, copy to your project:")
print("  best.pt   → Assignment 3/streamlit_app/assets/best.pt")
print("  best.onnx → Assignment 3/streamlit_app/assets/best.onnx")